In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
print("Libraries loaded")

In [ ]:
# Load full data (100%)
df = pd.read_parquet('processed/full_data.parquet', engine='fastparquet')
print(f"Full data shape: {df.shape}")

# Create features
customer_features = df.groupby('user_id').agg({
    'product_id': 'count',
    'order_number': 'max',
    'order_hour_of_day': 'mean',
    'order_dow': 'mean',
    'days_since_prior_order': 'mean',
}).reset_index()

customer_features.rename(columns={
    'product_id': 'total_products',
    'order_number': 'total_orders',
    'order_hour_of_day': 'avg_order_hour',
    'order_dow': 'avg_order_dow',
    'days_since_prior_order': 'avg_days_between_orders'
}, inplace=True)

# Cart size
cart_sizes = df.groupby(['user_id', 'order_id']).size().reset_index(name='cart_size')
avg_cart = cart_sizes.groupby('user_id')['cart_size'].mean().reset_index()
customer_features = customer_features.merge(avg_cart.rename(columns={'cart_size': 'avg_cart_size'}), on='user_id')

# Handle NaNs
customer_features['avg_days_between_orders'] = customer_features['avg_days_between_orders'].fillna(0)
print(f"Unique customers: {len(customer_features):,}")

In [ ]:
# Prepare features
X = customer_features[['total_products', 'total_orders', 'avg_order_hour', 'avg_order_dow', 'avg_cart_size', 'avg_days_between_orders']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled)

# Quick K-Means (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=3)
customer_features['segment'] = kmeans.fit_predict(X_scaled)

# Stats
segment_stats = customer_features.groupby('segment').agg({
    'total_products': 'mean',
    'total_orders': 'mean',
    'avg_order_hour': 'mean',
    'avg_order_dow': 'mean',
    'avg_cart_size': 'mean',
    'user_id': 'count'
}).rename(columns={'user_id': 'customer_count'})

print(segment_stats.round(2))

In [ ]:
# Profile mapping
segment_order = segment_stats.sort_values('total_orders', ascending=False).index.tolist()
profiles = ['Cart Addict', 'Weekend Buyer', 'Casual Buyer', 'Midnight Shopper']
segment_mapping = dict(zip(segment_order, profiles))
customer_features['profile'] = customer_features['segment'].map(segment_mapping)

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for seg in range(4):
    mask = customer_features['segment'] == seg
    axes[0].scatter(customer_features.loc[mask, 'total_orders'], customer_features.loc[mask, 'avg_order_hour'], c=colors[seg], label=segment_mapping.get(seg), alpha=0.5, s=10)
    axes[1].scatter(customer_features.loc[mask, 'total_products'], customer_features.loc[mask, 'avg_cart_size'], c=colors[seg], label=segment_mapping.get(seg), alpha=0.5, s=10)

axes[0].set_xlabel('Total Orders'); axes[0].set_ylabel('Avg Order Hour'); axes[0].set_title('Orders vs Hour'); axes[0].legend(); axes[0].grid(True)
axes[1].set_xlabel('Total Products'); axes[1].set_ylabel('Avg Cart Size'); axes[1].set_title('Products vs Cart Size'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig('customer_segmentation_scatter.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved: customer_segmentation_scatter.png")

# Department analysis (using full df)
df_with_segments = df.merge(customer_features[['user_id', 'segment', 'profile']], on='user_id')
dept_by_segment = df_with_segments.groupby(['profile', 'department']).size().unstack(fill_value=0)
dept_pct = dept_by_segment.div(dept_by_segment.sum(axis=1), axis=0) * 100
top_departments = df['department'].value_counts().head(8).index.tolist()
dept_pct_top = dept_pct[top_departments]

plt.figure(figsize=(12, 6))
dept_pct_top.T.plot(kind='bar')
plt.xticks(rotation=45, ha='right')
plt.title('Department Preferences by Segment')
plt.tight_layout()
plt.savefig('customer_segmentation_departments.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved: customer_segmentation_departments.png")

In [ ]:
# Save results
customer_features.to_parquet('processed/customer_segments.parquet', index=False)
print("Saved: processed/customer_segments.parquet")
print("\nSegmentation Complete!")